# Tempi e distanze delle relazioni origine-destinazione

Il notebook consolida le matrici provinciali del Friuli-Venezia Giulia e associa gli indicatori di tempo e distanza ai flussi di pendolarismo elaborati nel notebook `elaborazione_flussi.ipynb`.

## Legenda dei campi

| Campo | Descrizione |
|---|---|
| `OR_DEST` | Codice del comune di origine e del comune di destinazione |
| `OR_REG` | Codice della regione di origine |
| `OR_PROV` | Codice della provincia di origine |
| `OR_PROCOM` | Codice del comune di origine |
| `DEST_REG` | Codice della regione di destinazione |
| `DEST_PROV` | Codice della provincia di destinazione |
| `DEST_PROCOM` | Codice del comune di destinazione |
| `TEP_TOT` | Tempo effettivo di percorrenza totale, in minuti |
| `KM_TOT` | Distanza stradale individuata, in chilometri |
| `TTP_TOT` | Tempo teorico di percorrenza totale, in minuti |

Le unità e le modalità di calcolo sono documentate da ISTAT. Le distanze si riferiscono al percorso che minimizza il tempo e non necessariamente al percorso più breve. Le matrici usano la geografia comunale al 1° gennaio 2021, il grafo TomTom al 31 dicembre 2020 e lo scenario del 2 ottobre 2020 alle 08:30.

In [ ]:
from pathlib import Path

import pandas as pd

DIR_INPUT = Path("input")
DIR_OUTPUT = Path("output")
DIR_TEMPI_DISTANZE = DIR_INPUT / "tempi_distanze"
DIR_OUTPUT_INTRA = DIR_OUTPUT / "intra_regione"
DIR_OUTPUT_EXTRA = DIR_OUTPUT / "extra_regione"
DIR_OUTPUT_INTRA.mkdir(parents=True, exist_ok=True)
DIR_OUTPUT_EXTRA.mkdir(parents=True, exist_ok=True)

FILE_MATRICI = [
    DIR_TEMPI_DISTANZE / nome
    for nome in ["R06_GO.csv", "R06_PN.csv", "R06_TS.csv", "R06_UD.csv"]
]
FILE_FLUSSI_INTERNI = DIR_OUTPUT_INTRA / "flussi_mobilita_interni_FVG_2021.csv"
FILE_FLUSSI_EXTRA = DIR_OUTPUT_EXTRA / "flussi_mobilita_extra_regione_FVG_2021.csv"
FILE_OUTPUT_INTERNI = (
    DIR_OUTPUT_INTRA / "flussi_mobilita_interni_FVG_2021_tempi_distanze.csv"
)
FILE_OUTPUT_EXTRA = (
    DIR_OUTPUT_EXTRA / "flussi_mobilita_extra_regione_FVG_2021_tempi_distanze.csv"
)

colonne_codice = [
    "OR_REG", "OR_PROV", "OR_PROCOM",
    "DEST_REG", "DEST_PROV", "DEST_PROCOM",
]
tipi_colonne = {colonna: "string" for colonna in colonne_codice}


## Consolidamento e normalizzazione

I quattro file provinciali condividono lo stesso schema. Durante la lettura sono applicati il separatore a punto e virgola e la virgola decimale. I codici territoriali sono trattati come stringhe e completati con zeri iniziali per renderli compatibili con i codici della matrice di pendolarismo.

In [ ]:
matrici_provinciali = []
for file_matrice in FILE_MATRICI:
    matrice = pd.read_csv(
        file_matrice,
        sep=";",
        decimal=",",
        dtype=tipi_colonne,
    )
    # Alcune intestazioni della fonte contengono spazi terminali.
    matrice.columns = matrice.columns.str.strip()
    matrice["File_origine"] = file_matrice
    matrici_provinciali.append(matrice)

matrice_tempi_distanze = pd.concat(matrici_provinciali, ignore_index=True)

# Uniformazione della lunghezza dei codici ISTAT.
lunghezza_codici = {
    "OR_REG": 2, "OR_PROV": 3, "OR_PROCOM": 6,
    "DEST_REG": 2, "DEST_PROV": 3, "DEST_PROCOM": 6,
}
for colonna, lunghezza in lunghezza_codici.items():
    matrice_tempi_distanze[colonna] = (
        matrice_tempi_distanze[colonna].str.strip().str.zfill(lunghezza)
    )

# Conversione di tempo effettivo, distanza stradale e tempo teorico.
colonne_indicatori = ["TEP_TOT", "KM_TOT", "TTP_TOT"]
for colonna in colonne_indicatori:
    matrice_tempi_distanze[colonna] = pd.to_numeric(
        matrice_tempi_distanze[colonna], errors="coerce"
    )

# Ogni coppia origine-destinazione deve comparire una sola volta.
chiave_od = ["OR_PROCOM", "DEST_PROCOM"]
duplicati_od = matrice_tempi_distanze.duplicated(chiave_od, keep=False)
if duplicati_od.any():
    raise ValueError(
        f"La matrice contiene {int(duplicati_od.sum()):,} righe con chiavi OD duplicate."
    )

riepilogo_matrice = pd.Series({
    "Righe": len(matrice_tempi_distanze),
    "Origini": matrice_tempi_distanze["OR_PROCOM"].nunique(),
    "Destinazioni": matrice_tempi_distanze["DEST_PROCOM"].nunique(),
    "Chiavi OD duplicate": int(duplicati_od.sum()),
}, name="Matrice consolidata")
display(riepilogo_matrice.to_frame())

## Controllo degli indicatori

Il controllo riporta completezza e distribuzione degli indicatori. Valori nulli di tempo e distanza sono ammessi per gli archi intra-comunali; eventuali valori mancanti o negativi sono segnalati separatamente.

In [ ]:
controllo_indicatori = pd.DataFrame({
    "Valori_mancanti": matrice_tempi_distanze[colonne_indicatori].isna().sum(),
    "Valori_negativi": matrice_tempi_distanze[colonne_indicatori].lt(0).sum(),
    "Valori_nulli": matrice_tempi_distanze[colonne_indicatori].eq(0).sum(),
})
display(controllo_indicatori)
display(matrice_tempi_distanze[colonne_indicatori].describe())

if matrice_tempi_distanze[colonne_indicatori].isna().any().any():
    raise ValueError("La matrice contiene indicatori mancanti o non numerici.")
if matrice_tempi_distanze[colonne_indicatori].lt(0).any().any():
    raise ValueError("La matrice contiene indicatori negativi.")
autoarchi = matrice_tempi_distanze["OR_PROCOM"].eq(
    matrice_tempi_distanze["DEST_PROCOM"]
)
righe_con_zero = matrice_tempi_distanze[colonne_indicatori].eq(0).any(axis=1)
righe_tutte_zero = matrice_tempi_distanze[colonne_indicatori].eq(0).all(axis=1)
if (righe_con_zero != autoarchi).any() or (righe_tutte_zero != autoarchi).any():
    raise ValueError("I valori nulli non corrispondono esattamente agli autoarchi.")

## Analisi strutturale delle matrici

La consistenza di ciascun file è confrontata con il prodotto tra il numero di origini e il numero di destinazioni. L'uguaglianza tra righe osservate e righe attese indica che la relativa matrice provinciale è completa. Sono inoltre esaminate le correlazioni tra gli indicatori e le differenze tra i due versi delle relazioni interne al FVG.

In [ ]:
# Verifica della completezza cartesiana di ciascuna matrice provinciale.
riepilogo_file = (
    matrice_tempi_distanze.groupby("File_origine", as_index=False)
    .agg(
        Righe=("OR_PROCOM", "size"),
        Origini=("OR_PROCOM", "nunique"),
        Destinazioni=("DEST_PROCOM", "nunique"),
    )
)
riepilogo_file["Righe_attese"] = (
    riepilogo_file["Origini"] * riepilogo_file["Destinazioni"]
)
riepilogo_file["Matrice_completa"] = (
    riepilogo_file["Righe"] == riepilogo_file["Righe_attese"]
)
display(riepilogo_file)
if not riepilogo_file["Matrice_completa"].all():
    raise ValueError("Una o più matrici provinciali non sono cartesiane complete.")

# Correlazione lineare tra tempo effettivo, distanza e tempo teorico.
correlazioni_indicatori = matrice_tempi_distanze[colonne_indicatori].corr()
display(correlazioni_indicatori.round(4))

# Individuazione delle relazioni che presentano il massimo di ogni indicatore.
righe_massimi = []
for indicatore in colonne_indicatori:
    riga = (
        matrice_tempi_distanze.nlargest(1, indicatore)
        [["OR_PROCOM", "DEST_PROCOM"] + colonne_indicatori]
        .iloc[0]
        .to_dict()
    )
    riga["Indicatore_massimo"] = indicatore
    righe_massimi.append(riga)
massimi_indicatori = pd.DataFrame(righe_massimi)[
    ["Indicatore_massimo", "OR_PROCOM", "DEST_PROCOM"] + colonne_indicatori
]
display(massimi_indicatori)

# Confronto tra i due versi degli archi inter-comunali interni al FVG.
codici_origine_fvg = set(matrice_tempi_distanze["OR_PROCOM"].unique())
archi_fvg = matrice_tempi_distanze.loc[
    matrice_tempi_distanze["DEST_PROCOM"].isin(codici_origine_fvg),
    ["OR_PROCOM", "DEST_PROCOM"] + colonne_indicatori,
].copy()
archi_fvg = archi_fvg.loc[archi_fvg["OR_PROCOM"] != archi_fvg["DEST_PROCOM"]]

archi_fvg_inversi = archi_fvg.rename(columns={
    "OR_PROCOM": "DEST_PROCOM",
    "DEST_PROCOM": "OR_PROCOM",
    **{colonna: f"{colonna}_inverso" for colonna in colonne_indicatori},
})
confronto_versi = archi_fvg.merge(
    archi_fvg_inversi,
    on=["OR_PROCOM", "DEST_PROCOM"],
    validate="one_to_one",
)

statistiche_asimmetria = []
for indicatore in colonne_indicatori:
    differenza = (
        confronto_versi[indicatore]
        - confronto_versi[f"{indicatore}_inverso"]
    ).abs()
    statistiche_asimmetria.append({
        "Indicatore": indicatore,
        "Versi_identici_percentuale": round(differenza.eq(0).mean() * 100, 2),
        "Differenza_assoluta_mediana": differenza.median(),
        "Differenza_assoluta_95_percentile": differenza.quantile(0.95),
        "Differenza_assoluta_massima": differenza.max(),
    })
display(pd.DataFrame(statistiche_asimmetria))

## Associazione ai flussi di pendolarismo

Ogni riga finale conserva una relazione origine-destinazione ISTAT nel verso `comune di residenza → comune di lavoro`, con il relativo numero di `Pendolari`. Le relazioni opposte non sono sommate. Per i flussi extra-regionali sono mantenuti soltanto i residenti FVG che lavorano fuori regione.

In [ ]:
flussi_interni = pd.read_csv(
    FILE_FLUSSI_INTERNI,
    dtype={"Prov_res": "string", "Procom_res": "string",
           "Prov_lav": "string", "Procom_lav": "string",
           "Pendolari": "Int64"},
)
flussi_extra = pd.read_csv(
    FILE_FLUSSI_EXTRA,
    dtype={"Prov_res": "string", "Procom_res": "string",
           "Prov_lav": "string", "Procom_lav": "string",
           "Pendolari": "Int64"},
)

indicatori_od = matrice_tempi_distanze[
    ["OR_PROCOM", "DEST_PROCOM"] + colonne_indicatori
]

# La direzione ISTAT residenza -> lavoro è conservata senza riordinare la coppia.
direzione_ab = pd.Series(True, index=flussi_interni.index)
flussi_interni["Prov_A"] = flussi_interni["Prov_res"].where(
    direzione_ab, flussi_interni["Prov_lav"]
)
flussi_interni["Procom_A"] = flussi_interni["Procom_res"].where(
    direzione_ab, flussi_interni["Procom_lav"]
)
flussi_interni["Comune_A"] = flussi_interni["Comune_res"].where(
    direzione_ab, flussi_interni["Comune_lav"]
)
flussi_interni["Prov_B"] = flussi_interni["Prov_lav"].where(
    direzione_ab, flussi_interni["Prov_res"]
)
flussi_interni["Procom_B"] = flussi_interni["Procom_lav"].where(
    direzione_ab, flussi_interni["Procom_res"]
)
flussi_interni["Comune_B"] = flussi_interni["Comune_lav"].where(
    direzione_ab, flussi_interni["Comune_res"]
)
flussi_interni["Pendolari_A_B"] = flussi_interni["Pendolari"].where(
    direzione_ab, 0
)
flussi_interni["Pendolari_B_A"] = flussi_interni["Pendolari"].where(
    ~direzione_ab, 0
)

chiave_arco_interno = [
    "Prov_A", "Procom_A", "Comune_A",
    "Prov_B", "Procom_B", "Comune_B",
]
archi_interni = (
    flussi_interni.groupby(chiave_arco_interno, as_index=False)
    .agg(
        Pendolari_A_B=("Pendolari_A_B", "sum"),
        Pendolari_B_A=("Pendolari_B_A", "sum"),
    )
)
if archi_interni.duplicated(["Procom_A", "Procom_B"]).any():
    raise ValueError("Gli archi interni contengono chiavi A-B duplicate.")
if (
    archi_interni[["Pendolari_A_B", "Pendolari_B_A"]].to_numpy().sum()
    != flussi_interni["Pendolari"].sum()
):
    raise ValueError("Il totale dei pendolari interni non è stato conservato.")

# Gli indicatori dei due versi sono mantenuti separati.
indicatori_ab = indicatori_od.rename(columns={
    "OR_PROCOM": "Procom_A",
    "DEST_PROCOM": "Procom_B",
    **{colonna: f"{colonna}_A_B" for colonna in colonne_indicatori},
})
indicatori_ba = indicatori_od.rename(columns={
    "OR_PROCOM": "Procom_B",
    "DEST_PROCOM": "Procom_A",
    **{colonna: f"{colonna}_B_A" for colonna in colonne_indicatori},
})
flussi_interni_td = archi_interni.merge(
    indicatori_ab, how="left", on=["Procom_A", "Procom_B"],
    validate="one_to_one", indicator="_merge_A_B",
).merge(
    indicatori_ba, how="left", on=["Procom_A", "Procom_B"],
    validate="one_to_one", indicator="_merge_B_A",
)

# Sono mantenuti soltanto i residenti FVG che lavorano fuori regione.
flussi_extra = flussi_extra.loc[flussi_extra["Direzione"].eq("Uscita")].copy()
uscita = pd.Series(True, index=flussi_extra.index)
flussi_extra["Prov_FVG"] = flussi_extra["Prov_res"].where(
    uscita, flussi_extra["Prov_lav"]
)
flussi_extra["Procom_FVG"] = flussi_extra["Procom_res"].where(
    uscita, flussi_extra["Procom_lav"]
)
flussi_extra["Comune_FVG"] = flussi_extra["Comune_res"].where(
    uscita, flussi_extra["Comune_lav"]
)
flussi_extra["Prov_esterno"] = flussi_extra["Prov_lav"].where(
    uscita, flussi_extra["Prov_res"]
)
flussi_extra["Procom_esterno"] = flussi_extra["Procom_lav"].where(
    uscita, flussi_extra["Procom_res"]
)
flussi_extra["Comune_esterno"] = flussi_extra["Comune_lav"].where(
    uscita, flussi_extra["Comune_res"]
)
flussi_extra["Pendolari_uscita"] = flussi_extra["Pendolari"].where(uscita, 0)
flussi_extra["Pendolari_entrata"] = flussi_extra["Pendolari"].where(~uscita, 0)

chiave_arco_extra = [
    "Prov_FVG", "Procom_FVG", "Comune_FVG",
    "Prov_esterno", "Procom_esterno", "Comune_esterno",
]
archi_extra = (
    flussi_extra.groupby(chiave_arco_extra, as_index=False)
    .agg(
        Pendolari_uscita=("Pendolari_uscita", "sum"),
        Pendolari_entrata=("Pendolari_entrata", "sum"),
    )
)
if archi_extra.duplicated(["Procom_FVG", "Procom_esterno"]).any():
    raise ValueError("Gli archi extra-regionali contengono chiavi duplicate.")
if (
    archi_extra[["Pendolari_uscita", "Pendolari_entrata"]].to_numpy().sum()
    != flussi_extra["Pendolari"].sum()
):
    raise ValueError("Il totale dei pendolari extra-regionali non è stato conservato.")

# Gli indicatori sono sempre quelli del percorso FVG -> comune esterno.
flussi_extra_td = archi_extra.merge(
    indicatori_od,
    how="left",
    left_on=["Procom_FVG", "Procom_esterno"],
    right_on=["OR_PROCOM", "DEST_PROCOM"],
    validate="one_to_one",
    indicator=True,
).drop(columns=["OR_PROCOM", "DEST_PROCOM"])
flussi_extra_td["Orientamento_indicatori"] = "FVG -> comune esterno"
flussi_interni_td["Indicatori_disponibili"] = (
    flussi_interni_td["_merge_A_B"].eq("both")
    & flussi_interni_td["_merge_B_A"].eq("both")
)
flussi_extra_td["Indicatori_disponibili"] = flussi_extra_td["_merge"].eq("both")
if not flussi_interni_td["Indicatori_disponibili"].all():
    raise ValueError("Join incompleto per gli indicatori degli archi interni.")
if not flussi_extra_td["Indicatori_disponibili"].all():
    raise ValueError("Join incompleto per gli indicatori degli archi extra-regionali.")

indicatori_interni = [
    f"{indicatore}_{verso}"
    for verso in ["A_B", "B_A"]
    for indicatore in colonne_indicatori
]
if flussi_interni_td[indicatori_interni].isna().any().any():
    raise ValueError("Gli archi interni associati contengono indicatori mancanti.")
if flussi_extra_td[colonne_indicatori].isna().any().any():
    raise ValueError("Gli archi extra-regionali associati contengono indicatori mancanti.")
if flussi_interni_td[indicatori_interni].lt(0).any().any():
    raise ValueError("Gli archi interni associati contengono indicatori negativi.")
if flussi_extra_td[colonne_indicatori].lt(0).any().any():
    raise ValueError("Gli archi extra-regionali associati contengono indicatori negativi.")

copertura_join = pd.DataFrame({
    "Dataset": ["Archi interni", "Archi extra-regionali"],
    "Relazioni": [len(flussi_interni_td), len(flussi_extra_td)],
    "Relazioni_associate": [
        flussi_interni_td["Indicatori_disponibili"].sum(),
        flussi_extra_td["_merge"].eq("both").sum(),
    ],
    "Relazioni_non_associate": [
        (~flussi_interni_td["Indicatori_disponibili"]).sum(),
        flussi_extra_td["_merge"].ne("both").sum(),
    ],
})
display(copertura_join)

## Esportazione

I dataset arricchiti sono esportati mantenendo una riga per relazione direzionale residenza-lavoro e una sola colonna `Pendolari`. Le colonne tecniche e le rappresentazioni di controllo A-B sono escluse dai file finali.

In [ ]:
colonne_finali = [
    "Prov_res", "Procom_res", "Comune_res",
    "Prov_lav", "Procom_lav", "Comune_lav",
    "Pendolari", "TEP_TOT", "KM_TOT", "TTP_TOT",
]
dataset_interni_finale = flussi_interni_td.rename(columns={
    "Prov_A": "Prov_res", "Procom_A": "Procom_res",
    "Comune_A": "Comune_res", "Prov_B": "Prov_lav",
    "Procom_B": "Procom_lav", "Comune_B": "Comune_lav",
    "Pendolari_A_B": "Pendolari",
    "TEP_TOT_A_B": "TEP_TOT", "KM_TOT_A_B": "KM_TOT",
    "TTP_TOT_A_B": "TTP_TOT",
})[colonne_finali]
dataset_extra_finale = flussi_extra_td.rename(columns={
    "Prov_FVG": "Prov_res", "Procom_FVG": "Procom_res",
    "Comune_FVG": "Comune_res", "Prov_esterno": "Prov_lav",
    "Procom_esterno": "Procom_lav", "Comune_esterno": "Comune_lav",
    "Pendolari_uscita": "Pendolari",
})[colonne_finali]

dataset_interni_finale.to_csv(
    FILE_OUTPUT_INTERNI, index=False, encoding="utf-8-sig"
)
dataset_extra_finale.to_csv(
    FILE_OUTPUT_EXTRA, index=False, encoding="utf-8-sig"
)

riepilogo_output = pd.DataFrame({
    "File": [FILE_OUTPUT_INTERNI, FILE_OUTPUT_EXTRA],
    "Righe": [len(dataset_interni_finale), len(dataset_extra_finale)],
})
display(riepilogo_output)